# 04 — SmartPlate End-to-End Demo

**What this notebook does**
1. Loads the trained SegFormer-B0 checkpoint from `../checkpoints/segformer_best/`.
2. Runs inference on held-out validation images.
3. Aggregates predicted pixels → 7 Harvard food groups → 0–100 health score + macros.
4. Displays side-by-side: original image, ground-truth mask, predicted mask, score card.
5. Optionally runs on any custom photo you drop into `../assets/custom_plates/`.

## 1. Imports and paths

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import torch

sys.path.insert(0, os.path.abspath('..'))
from smartplate import (
    load_class_to_group, load_macros,
    mask_to_proportions, score_plate, proportions_to_macros,
    overlay_mask, predict_mask,
)

CHECKPOINT_DIR  = '../checkpoints/segformer_best'
GROUPS_CSV      = '../assets/foodseg103_to_groups.csv'
MACROS_CSV      = '../assets/usda_macros.csv'
FIGURES_DIR     = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Load model and mappings

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

processor = SegformerImageProcessor.from_pretrained(CHECKPOINT_DIR)
model     = SegformerForSemanticSegmentation.from_pretrained(CHECKPOINT_DIR).to(device)
model.eval()

class_to_group = load_class_to_group(GROUPS_CSV)
macros_table   = load_macros(MACROS_CSV)

NUM_LABELS = model.config.num_labels
FOODSEG103_CLASSES = list(model.config.id2label.values())

print(f'Model loaded: {NUM_LABELS} classes, {sum(p.numel() for p in model.parameters())/1e6:.1f} M params')
print(f'Food-group mappings: {len(class_to_group)} classes mapped')

## 3. Score card helper

Given a predicted mask, this builds the full pipeline output: proportions → score → macros.

In [ ]:
GROUP_COLORS = {
    'vegetables':        '#4CAF50',
    'fruits':            '#FF9800',
    'whole_grains':      '#8BC34A',
    'healthy_protein':   '#2196F3',
    'refined_grains':    '#FFC107',
    'red_processed_meat':'#F44336',
    'sugary_fatty':      '#9C27B0',
}

def run_pipeline(image: Image.Image):
    """image → (mask, proportions, score, macros)"""
    mask        = predict_mask(model, processor, image, device=device)
    proportions = mask_to_proportions(mask, class_to_group)
    score       = score_plate(proportions)
    macros      = proportions_to_macros(proportions, macros_table)
    return mask, proportions, score, macros


def plot_result(image, gt_mask, pred_mask, proportions, score, macros,
                title='', save_path=None):
    fig = plt.figure(figsize=(16, 9))
    gs  = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.3)

    # Row 0: image, GT mask, predicted mask
    ax_img  = fig.add_subplot(gs[0, 0])
    ax_gt   = fig.add_subplot(gs[0, 1])
    ax_pred = fig.add_subplot(gs[0, 2])
    ax_sc   = fig.add_subplot(gs[0, 3])

    ax_img.imshow(image);  ax_img.set_title('Input'); ax_img.axis('off')
    if gt_mask is not None:
        ax_gt.imshow(overlay_mask(image, gt_mask, num_classes=NUM_LABELS))
        ax_gt.set_title('Ground truth'); ax_gt.axis('off')
    else:
        ax_gt.axis('off'); ax_gt.set_title('')
    ax_pred.imshow(overlay_mask(image, pred_mask, num_classes=NUM_LABELS))
    ax_pred.set_title('Prediction'); ax_pred.axis('off')

    # Score gauge
    score_color = '#4CAF50' if score >= 70 else ('#FF9800' if score >= 45 else '#F44336')
    ax_sc.text(0.5, 0.55, f'{score:.0f}', ha='center', va='center',
               fontsize=52, fontweight='bold', color=score_color,
               transform=ax_sc.transAxes)
    ax_sc.text(0.5, 0.25, 'Health Score / 100', ha='center', va='center',
               fontsize=10, color='#555', transform=ax_sc.transAxes)
    ax_sc.axis('off')

    # Row 1: food-group bar chart + macros table
    ax_bar  = fig.add_subplot(gs[1, :2])
    ax_mac  = fig.add_subplot(gs[1, 2:])

    groups  = [g for g in GROUP_COLORS if proportions.get(g, 0) > 0]
    vals    = [proportions[g] * 100 for g in groups]
    colors  = [GROUP_COLORS[g] for g in groups]
    labels  = [g.replace('_', ' ') for g in groups]
    bars = ax_bar.barh(labels, vals, color=colors)
    ax_bar.set_xlabel('% of food area')
    ax_bar.set_title('Food group breakdown')
    ax_bar.set_xlim(0, 100)
    for bar, v in zip(bars, vals):
        ax_bar.text(v + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{v:.1f}%', va='center', fontsize=9)

    # Macros table
    ax_mac.axis('off')
    table_data = [
        ['Energy',   f'{macros["kcal"]:.0f} kcal'],
        ['Protein',  f'{macros["protein_g"]:.1f} g'],
        ['Carbs',    f'{macros["carb_g"]:.1f} g'],
        ['Fat',      f'{macros["fat_g"]:.1f} g'],
    ]
    tbl = ax_mac.table(cellText=table_data, colLabels=['Macro', 'Est. (400 g plate)'],
                       loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 2)
    ax_mac.set_title('Estimated macros', pad=12)

    if title:
        fig.suptitle(title, fontsize=13, y=1.01)
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches='tight')
    plt.show()
    return fig

print('Pipeline helpers ready.')

## 4. Run on held-out validation images

In [ ]:
from datasets import load_dataset

ds  = load_dataset('EduardoPacheco/FoodSeg103', split='validation')
rng = np.random.default_rng(7)
indices = rng.choice(len(ds), size=6, replace=False)

print(f'Running demo on {len(indices)} val images...')
for i, idx in enumerate(indices):
    ex      = ds[int(idx)]
    image   = ex['image'].convert('RGB')
    gt_mask = np.array(ex['label'])

    mask, proportions, score, macros = run_pipeline(image)

    present = [FOODSEG103_CLASSES[c] for c in sorted(set(gt_mask.flatten()) - {0})]
    print(f'\n[{i+1}] val #{idx} | score={score:.1f} | foods: {", ".join(present[:6])}')

    plot_result(
        image, gt_mask, mask, proportions, score, macros,
        title=f'val #{idx} — Health Score {score:.0f}/100',
        save_path=FIGURES_DIR / f'demo_val_{idx}.png',
    )

## 5. Run on a custom photo

Drop any food photo into `../assets/custom_plates/` and run the cell below.
No ground-truth mask needed — the model scores whatever it sees.

In [ ]:
custom_dir = Path('../assets/custom_plates')
custom_dir.mkdir(exist_ok=True)

custom_images = sorted(custom_dir.glob('*.jpg')) + sorted(custom_dir.glob('*.png'))

if not custom_images:
    print(f'No images found in {custom_dir}. Add a .jpg or .png file there and re-run.')
else:
    for img_path in custom_images:
        image = Image.open(img_path).convert('RGB')
        mask, proportions, score, macros = run_pipeline(image)
        print(f'{img_path.name}: score={score:.1f}/100')
        plot_result(
            image, None, mask, proportions, score, macros,
            title=f'{img_path.name} — Health Score {score:.0f}/100',
            save_path=FIGURES_DIR / f'demo_custom_{img_path.stem}.png',
        )

## 6. Score distribution across the full validation set

In [ ]:
from tqdm.auto import tqdm

N_SAMPLE = 200   # set to len(ds) for the full val set (~2135 images, ~10 min)
sample_idx = rng.choice(len(ds), size=N_SAMPLE, replace=False)

scores = []
for idx in tqdm(sample_idx, desc='scoring val images'):
    ex    = ds[int(idx)]
    image = ex['image'].convert('RGB')
    _, proportions, score, _ = run_pipeline(image)
    scores.append(score)

scores = np.array(scores)
print(f'n={N_SAMPLE}  mean={scores.mean():.1f}  median={np.median(scores):.1f}  '
      f'std={scores.std():.1f}  min={scores.min():.1f}  max={scores.max():.1f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=20, color='#2196F3', edgecolor='white')
ax.axvline(scores.mean(), color='#F44336', linestyle='--', label=f'mean={scores.mean():.1f}')
ax.set_xlabel('Health Score'); ax.set_ylabel('Count')
ax.set_title(f'Score distribution — {N_SAMPLE} validation images')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'score_distribution.png', dpi=140)
plt.show()